# 08 — Walk-Forward Backtest (การทดสอบแบบเลื่อนหน้าต่างเวลา)

รัน harness ของ Phase 8: หน้าต่าง **anchored walk-forward** (ไม่ใช่ random/k-fold เด็ดขาด — กฎ #6) ขับ
เครื่องคิดความเสี่ยง Phase 7 ทับสัญญาณ/การส่งคำสั่ง Phase 5 พร้อม **โมเดลต้นทุนจริง** (commission + slippage + spread)

**ข้อมูล:** อ่าน OHLCV จาก Market Data Engine (snapshot Parquet) เท่านั้น — ไม่ดึง tvkit

> **Data gate:** ยังไม่มี backfill TFEX 5 ปี และ engine ยังไม่มีข้อมูล TFEX — ตัวเลข exit-criteria
(expectancy หลังหักต้นทุน, drawdown ในงบ, regime stability) จึง **เลื่อนออกไป (data-gated)**;
โน้ตบุ๊กนี้สาธิต *กลไก* บนข้อมูล snapshot ที่มีในเครื่อง ไม่ได้อ้างขนาดผลลัพธ์จริง

In [ ]:
from functools import partial

import polars as pl

from tfex_s50_multi_tf_swing.backtest.data_source import build_execution_bars, load_continuous_frames
from tfex_s50_multi_tf_swing.backtest.walk_forward import run_walk_forward
from tfex_s50_multi_tf_swing.config.settings import get_settings
from tfex_s50_multi_tf_swing.data.store import ParquetStore
from tfex_s50_multi_tf_swing.execution.models import ExecutionConfig
from tfex_s50_multi_tf_swing.signals import strategy_a, strategy_b, strategy_c
from tfex_s50_multi_tf_swing.signals.inputs import build_signal_inputs

In [ ]:
# Load the engine/Parquet snapshot and build the aligned inputs + raw execution bars.
settings = get_settings()
store = ParquetStore(settings.data_dir)
frames = load_continuous_frames(store, with_4h=False)  # engine source declines 4h
sig_cfg = settings.signal_config()
inputs = build_signal_inputs(
    frames,
    regime_thresholds=settings.regime_thresholds(),
    bias_config=settings.bias_config(),
    signal_config=sig_cfg,
)
raw_bars = build_execution_bars(frames["5m"])  # stand-in for raw per-contract until TFEX backfill

detect = {
    "A": lambda df: strategy_a.to_signals(strategy_a.classify_frame(df, config=sig_cfg)),
    "B": lambda df: strategy_b.to_signals(strategy_b.classify_frame(df, config=sig_cfg)),
    "C": lambda df: strategy_c.to_signals(strategy_c.classify_frame(df, config=sig_cfg)),
}

In [ ]:
# Run the anchored walk-forward: combined (shared daily session) + per-strategy (isolated).
report = run_walk_forward(
    inputs=inputs,
    raw_bars=raw_bars,
    detect=detect,
    wf_config=settings.walk_forward_config(),
    exec_config=settings.execution_config(),
    risk_config=settings.risk_config(),
    cost_model=settings.cost_model(),
)
print(f"windows: {len(report.windows)}")
for sid, res in {"combined": report.combined, **report.per_strategy}.items():
    print(sid, "n_trades", res.overall.n_trades, "expectancy_r", float(res.overall.expectancy_r),
          "max_dd_r", float(res.drawdown.depth_r), "sharpe", res.ratios.sharpe)

## เส้นทุน (NAV indexed = 100) ต่อหน้าต่าง + drawdown

NAV ดัชนีฐาน 100 ต่อหน้าต่าง (เทียบ S50 buy-and-hold เป็น benchmark) และความลึก drawdown (R)

In [ ]:
# Per-window NAV index and drawdown depth (public-safe: counts/metrics, never raw OHLCV).
rows = [
    {"window": w.window.index, "nav_index": w.nav_index, "max_dd_r": float(w.drawdown.depth_r),
     "n_taken": w.n_taken, "regime_dominant": None}
    for w in report.combined.windows
]
nav_df = pl.DataFrame(rows)
print(nav_df)
if nav_df.height:
    _p = nav_df.to_pandas().set_index("window")
    _p["nav_index"].plot(title="Combined NAV index per window (base 100)", marker="o")

## Sensitivity sweep — ATR-stop multiplier

กวาดค่า `k_atr_stop` (ตัวคูณ stop ตาม ATR) 3 ค่าเพื่อดูความไวของ expectancy รวม

In [ ]:
# Sweep the most influential execution threshold; ML thresholds sweep similarly when enabled.
sweep = []
base_exec = settings.execution_config()
for k in (1.0, 1.5, 2.0):
    exec_cfg = ExecutionConfig(**{**base_exec.model_dump(), "k_atr_stop": k})
    rep = run_walk_forward(
        inputs=inputs, raw_bars=raw_bars, detect=detect,
        wf_config=settings.walk_forward_config(), exec_config=exec_cfg,
        risk_config=settings.risk_config(), cost_model=settings.cost_model(),
    )
    sweep.append({"k_atr_stop": k, "n_trades": rep.combined.overall.n_trades,
                  "expectancy_r": float(rep.combined.overall.expectancy_r)})
print(pl.DataFrame(sweep))

## สรุป

- harness, โมเดลต้นทุน, เมตริก (expectancy / drawdown profile / Sharpe-Sortino / regime concentration)
และรายงาน ทำงานครบ (machinery)
- เครื่องคิดความเสี่ยง Phase 7 ถูกขับต่อไม้จริงเป็นครั้งแรก (shared daily session ในรันรวม)
- **ตัวเลขผลลัพธ์จริงเลื่อนออกไป (data-gated)** จนกว่าจะมี backfill TFEX 5 ปี + ข้อมูล engine
- artifact สาธารณะเขียนโดย `scripts/run_walk_forward.py` → `results/static/backtest/walk_forward.json`
(นับ/เมตริกเท่านั้น ไม่มี OHLCV ดิบ)